In [1]:
import chromadb
import pandas as pd

from embeders import OllamaEmbeddingFunction

from tqdm import tqdm


In [2]:
client = chromadb.PersistentClient(
    path=r".\.chroma_db"
)

# Use Ollama for both add and query embeddings
ef = OllamaEmbeddingFunction(model="mxbai-embed-large", host="http://127.0.0.1:11434")
collection = client.get_or_create_collection(name="checks_v2", embedding_function=ef)


In [3]:
df_pubs = pd.read_excel("./2023_Completo_redem_0304.xlsx")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 64  # You can adjust this based on your resources

def process_batch(batch_df):
    messages = batch_df["Message"].tolist()
    res = collection.query(query_texts=messages, n_results=3)
    results = []
    for idx, message in enumerate(messages):
        distances = res["distances"][idx]
        min_distance = min(distances)
        results.append({
            "message": message,
            "min_distance": min_distance,
            "min_dist_document": res["documents"][idx][0],
            "doc_id": res["ids"][idx][0],
            "pub_id": batch_df.iloc[idx]["Message-ID"]
        })
    return results

df_res_list = []
batches = [df_pubs.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_pubs), BATCH_SIZE)]

with ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing batches"):
        df_res_list.extend(f.result())
        df_res = pd.DataFrame(df_res_list)
        df_res.to_csv("df_res.csv", index=False)


Processing batches:   5%|▍         | 263/5788 [1:31:21<35:13:08, 22.95s/it]